In [42]:
from pydantic import BaseModel, Field
from typing import List, Optional
from google import genai
from __future__ import annotations
import os
from dotenv import load_dotenv
import copy
import json
# from ollama import chat

# Schemas

### Exam Paper

In [43]:
class SubjectiveQuestion(BaseModel):
    question_id: str = Field(description="""
        Unique id for question in this format: 
        Q(n) where n is question number
        Q(n(m)) where n is parent question number while m is subquestion number
        and so on...
    """)
    marks: float = Field(description="Marks of the question or sum of marks of inner questions")
    question_statement: str = Field(description="""
        Statement or description the of the question. 
        It must not include sub questions statement (which will become part of "subquestions")
    """)
    subquestions: Optional[List[SubjectiveQuestion]] = Field(
        default=None, 
        description=""" 
        List of subquestions (Q(n(m)), Q(n(m(o))), ...) belonging directly to this question. 
        This field either has subquestions or is null/None (never empty list [])
        """
    )

SubjectiveQuestion.model_rebuild()

In [44]:
# class ObjectiveOption(BaseModel):
#     option: str = Field(description="Option identifier like: A, B, C... or i, ii, iii...")
#     value: str = Field(description="Option's value")

In [45]:
class ObjectiveQuestion(BaseModel):
    question_id: str = Field(description="""
        Unique id for question in this format: 
        Q(n) where n is question number
        Q(n(m)) where n is outer question number while m is inner question number
        and so on...
        """)
    marks: float = Field(description="Mark of the MCQ")
    mcq_statement: str = Field(description="Statement MCQ and its options")

In [46]:
class ExamPaper(BaseModel):
    subject : str = Field(description="Name of the subject")
    course_id: str | None = Field(default=None, description="ID of paper (only if available)")
    date: str | None = Field(default=None, description="Date of exam (only if available)")
    total_marks: float = Field(description="Total marks of all questions in the exam")
    objective: List[ObjectiveQuestion] | None = Field(default=None, description="Collection of Objective MCQs")
    subjective: List[SubjectiveQuestion] | None = Field(default=None, description="Collection of questions and sub questions")

ExamPaper.model_rebuild()

### Answer Paper

In [47]:
class ObjectiveAnswer(BaseModel):
    question_id: str = Field(description="""
        Unique id for question in this format Q(n) or Q(n(m)) or...
        Extract it from reference Exam paper
        """)
    chosen_option: str = Field(description="The identifier of option chosen for MCQ. If student selected more than one student, give ''.")

In [48]:
class SubjectiveAnswer(BaseModel):
    question_id: str = Field(description="""
        Unique id forq uestion in this format Q(n) or Q(n(m)) or...
        Extract it from reference Exam paper
    """)
    answer_statement: str | None = Field(default=None, description="Statement or text of the answer (A question will either have answer statement or sub answers)")
    subanswers: Optional[List[SubjectiveAnswer]] = Field(
        default=None, 
        description="""
            List of subanswers (Q(n(m)), Q(n(m(o))), ...) belonging directly to this question. 
            This field either has subquestions or is null/None (never empty list [])
            A question will either have answer statement or sub answers 
        """
    )

SubjectiveAnswer.model_rebuild()

In [49]:
class AnswerPaper(BaseModel):
    subject : str = Field(description="Name of the subject")
    roll_no: str = Field(description="Roll number of student")
    name: str | None = Field(default=None, description="Name of student (if available)")
    objective: List[ObjectiveAnswer] | None = Field(default=None, description="Collectection of MCQ chosen options")
    subjective : List[SubjectiveAnswer] | None = Field(default=None, description="Collection of answers and sub answers")

AnswerPaper.model_rebuild()

# Formatter

## LLM used

### Gemini

In [50]:
class ExamFormatterGemini:

    def __init__(self, client: genai.Client):
        self.client = client

    def format_exam_paper(self, exam_paper: str, model: str = "gemini-3.5-flash-lite"):
        response = self.client.interactions.create(
            model=model,
            input=f"""
                Format this exam according to given schema. 
                Carefully identify questions and their subquestions. 
                If a question contains sub-parts or sub-questions, you MUST populate the "subquestions" list with nested SubjectiveQuestion objects. 
                Do not leave subquestions empty or null if sub-parts are present in the text.
                You will encounter marks of question in front of them.
                Exam paper: {exam_paper}
                Ignore markdown tags, escape sequences & special symbols while not losing any question.
            """,
            response_format={
                "type": "text",
                "mime_type": "application/json",
                "schema": self.__inline_schema(ExamPaper)
            },
        )
        formatted_exam_paper = ExamPaper.model_validate_json(response.output_text)
        return formatted_exam_paper.model_dump_json()

    def format_answer_paper(self, answer_paper: str, exam_paper: str, model: str = "gemini-3.5-flash-lite"):
        response = self.client.interactions.create(
            model=model,
            input=f"""
                Format this answer paper according to given schema. 
                Carefully identify answers and their subanswers and format subanswer as part of "answers" of super answer.
                Exam paper is also attached so that you strictly follow the question ids and structure. The answer paper should have
                answers for all the questions & subquestions present in exam paper with same order and hierarchy even if empty.
                Exam paper: {exam_paper}
                Answer paper: {answer_paper}
                Ignore markdown tags, escape sequences & special symbols while not losing any question & its answer.
            """,
            response_format={
                "type": "text",
                "mime_type": "application/json",
                "schema": self.__inline_schema(AnswerPaper, "subanswers")
            },
        )
        formatted_answer_paper = AnswerPaper.model_validate_json(response.output_text)
        return formatted_answer_paper.model_dump_json()

    # Gemini isn't able to pick up recursive schema and it omits the nested levels
    # We define new schema which creates four levels of original recursive schema 
    # so now model reliably produced results for four levels of recursion
    def __inline_schema(self, model, property_name="subquestions", depth=4):
        schema = model.model_json_schema()
        defs = schema.pop("$defs", {})

        def resolve(node, d):
            if isinstance(node, dict):
                if "$ref" in node:
                    target = copy.deepcopy(defs[node["$ref"].split("/")[-1]])
                    if d <= 1:  # last level: no further nesting
                        target.get("properties", {}).pop(property_name, None)
                    return resolve(target, d - 1)
                return {k: resolve(v, d) for k, v in node.items()}
            if isinstance(node, list):
                return [resolve(i, d) for i in node]
            return node

        return resolve(schema, depth)

### Qwen - Ollama

In [51]:
class ExamFormatterQwen:

    def format_exam_paper(self, exam_paper: str, model: str = "qwen3:1.7b"):
        response = chat(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": f"""
                        Format this exam according to given schema. 
                        Carefully identify questions and their subquestions. Make subquestions part of "questions" of parent question.
                        You will encounter marks of question in front of them.
                        Exam paper: {exam_paper}
                    """
                }
            ],
            format=ExamPaper.model_json_schema(),
            options={'temperature': 0.2}
        )
        formatted_exam_paper = ExamPaper.model_validate_json(response.message.content)
        return formatted_exam_paper.model_dump_json()

    def format_answer_paper(self, answer_paper: str, exam_paper: str, model: str = "qwen3:1.7b"):
        response = chat(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": f"""
                        Format this answer paper according to given schema. 
                        Carefully identify answers and their subanswers and format subanswer as part of "answers" of super answer.
                        Exam paper is also attached so that you strictly follow the question ids, structure and sequence of questions, subquestions 
                        even if in answer paper, some questions are missing or reordered.
                        Exam paper: {exam_paper}
                        Answer paper: {answer_paper}
                    """,
                }
            ],
            format=ExamPaper.model_json_schema(),
            options={'temperature': 0.2}
        )
        formatted_answer_paper = AnswerPaper.model_validate_json(response.message.content)
        return formatted_answer_paper.model_dump_json()

## Testing

## Gemini

### Exam Paper

In [52]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

In [53]:
exam_formatter = ExamFormatterGemini(client)

In [54]:
exam_text = """
DATA STRUCTURES & ALGORITHMS — Mid-Term Examination
Time: 90 Minutes | Total Marks: 50

Instructions:
1. Two sections: A (Objective) and B (Subjective).
2. Marks for each question are written in front of it.
3. Section A contains MCQs only.
4. Answer Section B in your own words.

SECTION A — OBJECTIVE (MCQ) [10 x 2 = 20 Marks]

Q1. [2 marks] What is the worst-case time complexity of binary search on a sorted array of n elements?
(A) O(n) (B) O(log n) (C) O(n log n) (D) O(1)

Q2. [2 marks] Which data structure uses LIFO (Last In First Out) order?
(A) Queue (B) Stack (C) Linked List (D) Tree

Q3. [2 marks] In a max-heap containing n elements, where is the largest element always located?
(A) At the last leaf node (B) At the root (C) At any leaf node (D) At the middle index

Q4. [2 marks] What is the time complexity of inserting an element at the beginning of a singly linked list?
(A) O(1) (B) O(n) (C) O(log n) (D) O(n^2)

Q5. [2 marks] Which traversal of a Binary Search Tree visits nodes in ascending sorted order?
(A) Pre-order (B) Post-order (C) In-order (D) Level-order

Q6. [2 marks] Which sorting algorithm has the best average-case time complexity?
(A) Bubble Sort (B) Insertion Sort (C) Merge Sort (D) Selection Sort

Q7. [2 marks] A graph with no cycles and n-1 edges connecting n vertices is called a:
(A) Complete graph (B) Tree (C) Bipartite graph (D) Directed acyclic multigraph

Q8. [2 marks] Which data structure is most suitable for implementing a priority queue efficiently?
(A) Array (B) Linked List (C) Heap (D) Stack

Q9. [2 marks] What is the space complexity of a recursive Fibonacci function without memoization (in terms of call stack depth for input n)?
(A) O(1) (B) O(n) (C) O(log n) (D) O(2^n)

Q10. [2 marks] Which technique does Dijkstra's algorithm use to find the shortest path?
(A) Dynamic Programming only (B) Greedy approach (C) Backtracking (D) Divide and Conquer

SECTION B — SUBJECTIVE [5 x 4 + 1 x 10 = 30 Marks]

Q11. [4 marks] Differentiate between an array and a linked list. Give one real-world scenario where a linked list would be preferred over an array.

Q12. [4 marks] Explain what a hash collision is and briefly describe one method used to resolve it.

Q13. [4 marks] Write the recurrence relation for the Merge Sort algorithm and state its time complexity using this relation.

Q14. [4 marks] What is the difference between BFS (Breadth-First Search) and DFS (Depth-First Search)? Mention one use case for each.

Q15. [4 marks] Define "dynamic programming" and explain how it differs from plain recursion.

Q16. [10 marks] Consider an unsorted array of integers: [38, 27, 43, 3, 9, 82, 10].
(a) Trace through the Merge Sort algorithm step-by-step to sort this array, showing the divide and merge steps. (6 marks)
(b) State the time complexity of Merge Sort in the best, average, and worst case, and briefly justify why all three cases are the same. (4 marks)"""

In [55]:
formatted_exam = exam_formatter.format_exam_paper(exam_text)

In [56]:
json.loads(formatted_exam)

{'subject': 'DATA STRUCTURES & ALGORITHMS',
 'course_id': None,
 'date': None,
 'total_marks': 50.0,
 'objective': [{'question_id': 'Q1',
   'marks': 2.0,
   'mcq_statement': 'What is the worst-case time complexity of binary search on a sorted array of n elements?\n(A) O(n) (B) O(log n) (C) O(n log n) (D) O(1)'},
  {'question_id': 'Q2',
   'marks': 2.0,
   'mcq_statement': 'Which data structure uses LIFO (Last In First Out) order?\n(A) Queue (B) Stack (C) Linked List (D) Tree'},
  {'question_id': 'Q3',
   'marks': 2.0,
   'mcq_statement': 'In a max-heap containing n elements, where is the largest element always located?\n(A) At the last leaf node (B) At the root (C) At any leaf node (D) At the middle index'},
  {'question_id': 'Q4',
   'marks': 2.0,
   'mcq_statement': 'What is the time complexity of inserting an element at the beginning of a singly linked list?\n(A) O(1) (B) O(n) (C) O(log n) (D) O(n^2)'},
  {'question_id': 'Q5',
   'marks': 2.0,
   'mcq_statement': 'Which traversal o

### Answer paper

In [57]:
answer_paper = """
SSTUDENT ANSWER SHEET
Name: Sara Malik | Roll No: DSA-033

Q1: B
Q2: C
Q3: B
Q4: B
Q5: C
Q6: B
Q7: B
Q8: C
Q9: D
Q10: B

Q14. Ans: BFS uses a queue and explores all neighbors at the current depth before going deeper, so it is used to find the shortest path in an unweighted graph. DFS uses a stack or recursion and explores as far down one path as possible before backtracking, so it is used for topological sorting and detecting cycles in a graph.

Q11. Ans: An array is a fixed-size contiguous block of memory allowing O(1) random access but O(n) insertion/deletion in the middle. A linked list is a dynamic collection of nodes connected via pointers, giving O(1) insertion/deletion at the head but O(n) access time. Linked lists are preferred for implementing something like a browser's history (back/forward navigation) where frequent insertion and deletion happens at the ends.

Q16. Ans: (a) [38, 27, 43, 3, 9, 82, 10] is split into [38, 27, 43] and [3, 9, 82, 10]. The first splits into [38] and [27, 43], which further splits into [27] and [43]. Merging [27] and [43] gives [27, 43], then merging with [38] gives [27, 38, 43]. The second half [3, 9, 82, 10] splits into [3, 9] and [82, 10]. [3, 9] merges to [3, 9] and [82, 10] merges to [10, 82]. Merging [3, 9] and [10, 82] gives [3, 9, 10, 82]. Finally merging [27, 38, 43] and [3, 9, 10, 82] gives [3, 9, 10, 27, 38, 43, 82].
(b) Best, average and worst case are all O(n log n) because merge sort always divides the array exactly in half regardless of the values or their order, and the merge step always takes linear time O(n), so the total work per level is O(n) across log n levels no matter how the input is arranged.

Q13. Ans: T(n) = 2T(n/2) + O(n), which by the master theorem gives O(n log n).

"""

In [58]:
formatted_answer_paper = exam_formatter.format_answer_paper(answer_paper, formatted_exam)

In [59]:
json.loads(formatted_answer_paper)

{'subject': 'DATA STRUCTURES & ALGORITHMS',
 'roll_no': 'DSA-033',
 'name': 'Sara Malik',
 'objective': [{'question_id': 'Q1', 'chosen_option': 'B'},
  {'question_id': 'Q2', 'chosen_option': 'C'},
  {'question_id': 'Q3', 'chosen_option': 'B'},
  {'question_id': 'Q4', 'chosen_option': 'B'},
  {'question_id': 'Q5', 'chosen_option': 'C'},
  {'question_id': 'Q6', 'chosen_option': 'B'},
  {'question_id': 'Q7', 'chosen_option': 'B'},
  {'question_id': 'Q8', 'chosen_option': 'C'},
  {'question_id': 'Q9', 'chosen_option': 'D'},
  {'question_id': 'Q10', 'chosen_option': 'B'}],
 'subjective': [{'question_id': 'Q11',
   'answer_statement': "An array is a fixed-size contiguous block of memory allowing O(1) random access but O(n) insertion/deletion in the middle. A linked list is a dynamic collection of nodes connected via pointers, giving O(1) insertion/deletion at the head but O(n) access time. Linked lists are preferred for implementing something like a browser's history (back/forward navigation

## Qwen

In [ ]:
exam_formatter_qwen = ExamFormatterQwen()

In [ ]:
qwen_formatted_exam = exam_formatter_qwen.format_exam_paper(exam_text)

In [ ]:
qwen_formatted_exam

In [ ]:
qwen_formatted_answer = exam_formatter_qwen.format_answer_paper(answer_paper)

In [ ]:
qwen_formatted_answer

The current hardware of my system is not able to run Qwen reliably so choosing Gemini API.